In [0]:
#%pip install -r ../requirements.txt

In [0]:
#%pip install --upgrade numpy tensorflow

In [0]:
#%restart_python

In [0]:
import json
from tqdm import tqdm
from openai import OpenAI
import os
import mlflow
mlflow.tracing.disable()

#from transformers import pipeline, AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer


In [0]:


# How to get your Databricks token: https://docs.databricks.com/en/dev-tools/auth/pat.html
#DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')
# Alternatively in a Databricks notebook you can use this:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
  api_key=DATABRICKS_TOKEN,
  base_url="https://adb-4424763777139025.5.azuredatabricks.net/serving-endpoints"
)
#

In [0]:
MODEL_NAME = "databricks-meta-llama-3-1-8b-instruct"
model_id = "databricks-meta-llama-3-1-8b-instruct"

file_name = "../data/rebuilding_milo_chunks_docling_max_tokens128_min_tokens50_meta_llama3p18B.txt" 


In [0]:
def get_text():
    with open(file_name, 'r') as f:
        llama_chunks = f.readlines()
    return llama_chunks 
    
def get_samples(n_chunks_intervals=None):
    text_chunks = get_text()
    raw_outputs = []
    samples = text_chunks[:] if n_chunks_intervals == None else text_chunks[n_chunks_intervals[0]:n_chunks_intervals[1]]
    return samples

samples = get_samples()


In [0]:
#get_prompt(samples[29])

In [0]:
def get_prompt(text):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise and helpful medical tutor. "
                "Based on the provided text, generate a JSON object with exactly ONE question (as 'instruction') and ONE answer (as 'output').\n\n"
                "- The content must relate to health, exercise, sports, fitness, or physiotherapy.\n"
                "- Do not include multiple questions or answers.\n"
                "- Do not repeat the instruction in the output.\n"
                "- The output must contain a thorough and detailed, multi-paragraph question (as 'instruction') and answer (as 'output').\n"
                "- If the text is not relevant, return: {\"instruction\": \"NULL\", \"output\": \"NULL\"}\n\n"
                "- Respond ONLY with the JSON object. Do NOT include any explanation or commentary."
            ), #multi-paragraph answer
        },
        {
            "role": "user",
            "content": text.strip()
        },
    ]
    return messages

In [0]:
len(samples)

In [0]:
samples[1710]

# Compute and log with MLFlow the QA raw file

In [0]:
import json
from tqdm import tqdm
import time
import mlflow
import numpy as np
from datetime import datetime

# ==== PARAMETERS ====
batch_size = 5         # samples per batch
n_reps = 3             # completions per sample
temperature = 0.75
top_p = 0.65
sample_i = 1
sample_f = 500
max_tokens=384

json_output_name = f"synthetic_QA_{datetime.now().strftime('%Y%m%d_%H%M%S')}_samples{sample_i}-{sample_f}.json"
results = []

# Helper to chunk a list
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

# ==== START A SINGLE MLflow RUN ====
mlflow.set_experiment("../../logs/mlruns/qa_generation_experiment")

with mlflow.start_run(run_name=f"qa_batch_{sample_i}_{sample_f}"):

    # --- Log run-level parameters ---
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("temperature", temperature)
    mlflow.log_param("top_p", top_p)
    mlflow.log_param("max_tokens", max_tokens)
    mlflow.log_param("batch_size", batch_size)
    mlflow.log_param("n_reps", n_reps)
    mlflow.log_param("sample_range", f"{sample_i}-{sample_f}")

    total_prompt_tokens, total_completion_tokens = [], []

    # ==== MAIN GENERATION LOOP ====
    for batch in tqdm(chunk_list(samples[sample_i:sample_f], batch_size)):
        batch_prompts = [get_prompt(sample) for sample in batch]

        # Iterate over each sample in the batch
        for prompt in batch_prompts:
            for _ in range(n_reps):
                try:
                    chat_completion = client.chat.completions.create(
                        messages=prompt,
                        model=MODEL_NAME,
                        max_tokens=max_tokens,
                        temperature=temperature,
                        top_p=top_p,
                        n=1  # # one completion per API call; we repeat for n_reps
                    )

                    # Parse output
                    content = chat_completion.choices[0].message.content
                    usage = getattr(chat_completion, "usage", None)

                    try:
                        parsed = json.loads(content)
                        instruction = parsed.get("instruction", "")
                        output = parsed.get("output", "")
                    except json.JSONDecodeError:
                        instruction, output = None, content

                    results.append({
                        "prompt": prompt,
                        "instruction": instruction,
                        "output": output,
                        "usage": {
                            "prompt_tokens": getattr(usage, "prompt_tokens", None),
                            "completion_tokens": getattr(usage, "completion_tokens", None),
                            "total_tokens": getattr(usage, "total_tokens", None)
                        }
                    })

                    # Collect token metrics
                    if usage:
                        total_prompt_tokens.append(usage.prompt_tokens)
                        total_completion_tokens.append(usage.completion_tokens)

                    time.sleep(0.5)  # avoid hitting rate limits

                except Exception as e:
                    print(f"Error generating completion for prompt: {prompt}")
                    print(e)

    # ==== SAVE OUTPUT LOCALLY ====
    with open(json_output_name, "w") as f:
        json.dump(results, f, indent=2)

    # ==== LOG METRICS & ARTIFACTS ====
    if total_prompt_tokens:
        mlflow.log_metric("avg_prompt_tokens", np.mean(total_prompt_tokens))
        mlflow.log_metric("avg_completion_tokens", np.mean(total_completion_tokens))
        mlflow.log_metric("total_generations", len(results))

    # Upload JSON output as artifact
    mlflow.log_artifact(json_output_name)

    mlflow.set_tag("dataset", "synthetic_QA_physio")
    mlflow.set_tag("run_type", "generation")
    mlflow.set_tag("status", "completed")

print(f"✅ Generation completed. Logged results in MLflow experiment.")


In [0]:
bfcwjebfvbekfbrekjvnkervk,renf

In [0]:
results

# Compute QA raw file

In [0]:
import json
from tqdm import tqdm
import time

results = []

batch_size = 5  # number of samples per batch
n_reps = 3      # number of completions per sample
temperature = 0.75
top_p = 0.65
sample_i = 500
sample_f = 1000


# Helper to chunk a list
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

for batch in tqdm(chunk_list(samples[sample_i:sample_f], batch_size)):
    # Build batch prompts
    batch_prompts = [get_prompt(sample) for sample in batch]

    # Iterate over each sample in the batch
    for prompt in batch_prompts:
        for _ in range(n_reps):
            try:
                chat_completion = client.chat.completions.create(
                    messages=prompt,
                    model=MODEL_NAME,
                    max_tokens=256,
                    temperature=temperature,
                    top_p=top_p,
                    n=1  # one completion per API call; we repeat for n_reps
                )
                content = chat_completion.choices[0].message.content

                try:
                    parsed = json.loads(content)
                    instruction = parsed.get("instruction", "")
                    output = parsed.get("output", "")
                except json.JSONDecodeError:
                    instruction, output = None, content

                results.append({
                    "prompt": prompt,
                    "instruction": instruction,
                    "output": output
                })

                # Optional: short sleep to avoid hitting rate limits
                time.sleep(0.5)

            except Exception as e:
                print(f"Error generating completion for prompt: {prompt}")
                print(e)

json_output_name= f"test_17-10-25-100to500"
with open(f"{json_output_name}.json", "w") as f:
    json.dump(results, f)



# Test different prompt outputs

In [0]:
results = []
n=29
prompt = get_prompt(samples[n])
chat_completion = client.chat.completions.create(
    messages=prompt,
    model=MODEL_NAME,
    max_tokens=384,
    temperature=0.75,         
    top_p=0.65,
    n=3,              
)

for choice in chat_completion.choices:
    content = choice.message.content

    try:
        parsed = json.loads(content)
        instruction = parsed.get("instruction", "")
        output = parsed.get("output", "")
    except json.JSONDecodeError:
        instruction, output = None, content

    results.append({
        "prompt": prompt,
        "instruction": instruction,
        "output": output
    })


In [0]:
def get_key(d, n, key):
    print(d[n][key])

for n in range(0,3):
    if n%3==0:
        print('*'*15,int(n/3),'*'*15)
        print(results[n]['prompt'][1]['content'])
        
    print('-'*15, n, '-'*15)
    get_key(results, n, "instruction")
    get_key(results, n, "output")

In [0]:
results[1]

In [0]:
import json
from tqdm import tqdm
import time

results = []

batch_size = 5  # number of samples per batch
n_reps = 3      # number of completions per sample
temperature = 0.75
top_p = 0.65
sample_i = 500
sample_f = 1000


# Helper to chunk a list
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

for batch in tqdm(chunk_list(samples[sample_i:sample_f], batch_size)):
    # Build batch prompts
    batch_prompts = [get_prompt(sample) for sample in batch]

    # Iterate over each sample in the batch
    for prompt in batch_prompts:
        for _ in range(n_reps):
            try:
                chat_completion = client.chat.completions.create(
                    messages=prompt,
                    model=MODEL_NAME,
                    max_tokens=256,
                    temperature=temperature,
                    top_p=top_p,
                    n=1  # one completion per API call; we repeat for n_reps
                )
                content = chat_completion.choices[0].message.content

                try:
                    parsed = json.loads(content)
                    instruction = parsed.get("instruction", "")
                    output = parsed.get("output", "")
                except json.JSONDecodeError:
                    instruction, output = None, content

                results.append({
                    "prompt": prompt,
                    "instruction": instruction,
                    "output": output
                })

                # Optional: short sleep to avoid hitting rate limits
                time.sleep(0.5)

            except Exception as e:
                print(f"Error generating completion for prompt: {prompt}")
                print(e)

json_output_name= f"test_16-10-25-100to500"
with open(f"{json_output_name}.json", "w") as f:
    json.dump(results, f)

